# RDF-from-Images Evaluation (Bridge + Multi‑Judge)

This notebook evaluates RDF/Turtle outputs extracted from images **without gold labels** using two LLM-as-a-judge strategies:

- **Bridge evaluation**: first build a *reference graph* from the image (nodes/edges in JSON), then score RDF against that reference (plus optional LLM scoring).
- **Multi‑judge evaluation**: ask multiple judges (different models and/or temperatures) and aggregate scores (median/majority + disagreement stats).

You can run this on Kaggle or Colab. Just set:
- `GEMINI_API_KEY`
- `IMAGES_ROOT` (folder that contains your dataset images)
- the paths to your extraction JSON files (Zero/One/Few-shot).

**Outputs**
- `evaluation_results.csv`
- `evaluation_results.json`
- cached bridge references in `bridge_cache/`


In [ ]:
# --- Install deps (uncomment if needed) ---
# !pip -q install rdflib rapidfuzz tqdm pandas


In [ ]:
import os
import json
import base64
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Any, Optional, Tuple

import requests
import pandas as pd
from tqdm.auto import tqdm

from rdflib import Graph
from rdflib.namespace import SKOS
from rapidfuzz import fuzz


In [ ]:
# =========================
# 1) CONFIG
# =========================

# (A) API key
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "").strip()
assert GEMINI_API_KEY, "Set GEMINI_API_KEY env var (or set it here) before running."

# (B) Where your images live (folder that contains your dataset images)
# Example (Colab): "/content/drive/MyDrive/Research/.../Datasets/Data"
# Example (Kaggle): "/kaggle/input/soil-helth-data/Soil Health"
IMAGES_ROOT = os.getenv("IMAGES_ROOT", "").strip()
assert IMAGES_ROOT, "Set IMAGES_ROOT to the folder that contains your images."

# (C) Your extraction outputs (edit paths)
ZERO_SHOT_JSON = os.getenv("ZERO_SHOT_JSON", "rdf_ZeroShot_extractions.json")
ONE_SHOT_JSON  = os.getenv("ONE_SHOT_JSON",  "rdf_extractions_oneShot.json")
FEW_SHOT_JSON  = os.getenv("FEW_SHOT_JSON",  "rdf_extractions_fewShot.json")

# (D) Bridge cache (image -> reference graph JSON)
BRIDGE_CACHE_DIR = Path("bridge_cache")
BRIDGE_CACHE_DIR.mkdir(exist_ok=True, parents=True)

# (E) Judge configs
# You can add/remove models, or vary temperatures to get diverse judges.
JUDGES = [
    {"name": "judge_flash_t01", "model": "gemini-2.5-flash", "temperature": 0.1},
    {"name": "judge_flash_t07", "model": "gemini-2.5-flash", "temperature": 0.7},
    # If you have access, Pro can be stronger for judging:
    # {"name": "judge_pro_t01", "model": "gemini-2.5-pro", "temperature": 0.1},
]

# How strict fuzzy matching should be when aligning labels
LABEL_MATCH_THRESHOLD = 88  # 0..100 (higher = stricter)

# Weights for deterministic bridge scoring
WEIGHTS = {
    "syntax_ok": 0.15,
    "node_f1":   0.35,
    "edge_f1":   0.45,
    "halluc_penalty": 0.05,
}

print("Config OK.")


In [ ]:
# =========================
# 2) Gemini REST helpers
# =========================

def _gemini_endpoint(model: str) -> str:
    return f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={GEMINI_API_KEY}"

def encode_image_to_base64(image_path: str) -> str:
    data = Path(image_path).read_bytes()
    return base64.b64encode(data).decode("utf-8")

def gemini_generate(
    model: str,
    parts: List[Dict[str, Any]],
    temperature: float = 0.2,
    max_output_tokens: int = 4096,
) -> str:
    payload = {
        "contents": [{"role": "user", "parts": parts}],
        "generationConfig": {
            "temperature": temperature,
            "topP": 0.9,
            "maxOutputTokens": max_output_tokens,
        },
    }
    resp = requests.post(_gemini_endpoint(model), headers={"Content-Type": "application/json"}, data=json.dumps(payload))
    resp.raise_for_status()
    data = resp.json()
    return data["candidates"][0]["content"]["parts"][0]["text"]

def gemini_text(model: str, prompt: str, temperature: float = 0.2, max_output_tokens: int = 2048) -> str:
    return gemini_generate(model=model, parts=[{"text": prompt}], temperature=temperature, max_output_tokens=max_output_tokens)

def gemini_multimodal(model: str, image_path: str, prompt: str, temperature: float = 0.2, max_output_tokens: int = 4096) -> str:
    b64 = encode_image_to_base64(image_path)
    parts = [
        {"inlineData": {"mimeType": "image/jpeg", "data": b64}},
        {"text": prompt},
    ]
    return gemini_generate(model=model, parts=parts, temperature=temperature, max_output_tokens=max_output_tokens)


In [ ]:
# =========================
# 3) Load your extraction JSONs
# =========================

def load_extractions(json_path: str) -> pd.DataFrame:
    p = Path(json_path)
    assert p.exists(), f"File not found: {json_path}"
    obj = json.loads(p.read_text(encoding="utf-8"))
    rows = obj["dataset"]
    df = pd.DataFrame(rows)
    # Normalize key for joining across runs
    df["image_key"] = df["source_image"].apply(lambda x: os.path.basename(str(x)))
    return df

df_zero = load_extractions(ZERO_SHOT_JSON).assign(run="zero_shot")
df_one  = load_extractions(ONE_SHOT_JSON).assign(run="one_shot")
df_few  = load_extractions(FEW_SHOT_JSON).assign(run="few_shot")

df_all = pd.concat([df_zero, df_one, df_few], ignore_index=True)
print(df_all.shape, df_all["run"].value_counts().to_dict())
df_all.head()


In [ ]:
# =========================
# 4) Resolve image paths
# =========================

def resolve_image_path(source_image: str) -> str:
    # If the path in JSON exists, use it
    if source_image and os.path.exists(source_image):
        return source_image
    # Otherwise try within IMAGES_ROOT using basename
    candidate = os.path.join(IMAGES_ROOT, os.path.basename(str(source_image)))
    if os.path.exists(candidate):
        return candidate
    # Or try searching recursively (slower but robust)
    for root, _, files in os.walk(IMAGES_ROOT):
        if os.path.basename(str(source_image)) in files:
            return os.path.join(root, os.path.basename(str(source_image)))
    raise FileNotFoundError(f"Could not resolve image path for: {source_image}")

# Quick sanity check on a few images
for k in df_all["source_image"].head(3).tolist():
    print(k, "->", resolve_image_path(k))


In [ ]:
# =========================
# 5) RDF/Turtle parsing + extraction of a simple fact graph
# =========================

@dataclass
class ParsedRDF:
    syntax_ok: bool
    n_triples: int
    concepts: List[str]              # URIs as strings
    labels: Dict[str, str]           # URI -> prefLabel
    edges: List[Tuple[str,str,str]]  # (s_uri, p_uri, o_uri/lit)

def parse_turtle(turtle_text: str) -> ParsedRDF:
    g = Graph()
    try:
        g.parse(data=turtle_text, format="turtle")
        syntax_ok = True
    except Exception:
        return ParsedRDF(False, 0, [], {}, [])

    concepts = set()
    labels = {}
    edges = []

    for s, p, o in g:
        su, pu = str(s), str(p)
        ou = str(o)
        edges.append((su, pu, ou))
        concepts.add(su)
        if pu == str(SKOS.prefLabel):
            labels[su] = str(o).strip().lower()

    # Only keep concepts that are declared as skos:Concept if present
    typed_concepts = set(str(s) for s, p, o in g if str(p).endswith("type") and str(o) == str(SKOS.Concept))
    if typed_concepts:
        concepts = concepts.intersection(typed_concepts)

    return ParsedRDF(True, len(edges), sorted(concepts), labels, edges)

def rdf_edges_as_label_triples(parsed: ParsedRDF) -> List[Tuple[str,str,str]]:
    \"\"\"Convert RDF edges to (s_label, p_short, o_label) where possible.\"\"\"
    def short_pred(p: str) -> str:
        if "#" in p:
            return p.split("#")[-1]
        return p.rstrip("/").split("/")[-1]

    out = []
    for s, p, o in parsed.edges:
        s_lab = parsed.labels.get(s, s)
        o_lab = parsed.labels.get(o, o)
        out.append((s_lab, short_pred(p), o_lab))
    return out


In [ ]:
# =========================
# 6) BRIDGE STEP A: Build a reference graph from the IMAGE (cached)
# =========================
# Output JSON schema:
# { "nodes": [{"label": "..."}], "edges": [{"source": "...", "relation": "...", "target": "...", "direction": "->"}] }

BRIDGE_REFERENCE_PROMPT = \"\"\"
You are extracting a canonical diagram graph from an image.

Return ONLY valid JSON with this schema:
{
  "nodes": [{"label": "exact text on node, lowercase"}],
  "edges": [{"source": "node label", "relation": "edge meaning or label", "target": "node label", "direction": "->"}]
}

Rules:
- Node labels MUST be exact text from the image, but lowercase.
- Create one node per distinct box/circle label (ignore decorative titles like "figure x.y" unless it is inside a diagram node).
- For arrows/links: create edges with direction.
- If an arrow has no label, set relation to "arrow".
- If there is a branching or feedback loop, represent it with edges.
- DO NOT invent nodes or edges not visible in the image.
- Output JSON only. No markdown.
\"\"\"

def bridge_cache_path(image_path: str) -> Path:
    return BRIDGE_CACHE_DIR / (Path(image_path).name + ".bridge.json")

def get_bridge_reference(image_path: str, bridge_model: str = "gemini-2.5-flash") -> Dict[str, Any]:
    cache = bridge_cache_path(image_path)
    if cache.exists():
        return json.loads(cache.read_text(encoding="utf-8"))

    txt = gemini_multimodal(
        model=bridge_model,
        image_path=image_path,
        prompt=BRIDGE_REFERENCE_PROMPT,
        temperature=0.1,
        max_output_tokens=2048,
    )

    # Robust JSON parse
    try:
        ref = json.loads(txt)
    except json.JSONDecodeError:
        start = txt.find("{")
        end = txt.rfind("}")
        if start >= 0 and end > start:
            ref = json.loads(txt[start:end+1])
        else:
            raise

    cache.write_text(json.dumps(ref, indent=2, ensure_ascii=False), encoding="utf-8")
    return ref


In [ ]:
# =========================
# 7) BRIDGE STEP B: Deterministic scoring vs reference
# =========================

def _best_match(label: str, candidates: List[str]) -> Tuple[Optional[str], int]:
    if not candidates:
        return None, 0
    best = None
    best_score = -1
    for c in candidates:
        s = fuzz.token_sort_ratio(label, c)
        if s > best_score:
            best, best_score = c, s
    return best, int(best_score)

def align_labels(ref_labels: List[str], cand_labels: List[str], threshold: int = 88) -> Dict[str, Optional[str]]:
    mapping = {}
    for r in ref_labels:
        best, score = _best_match(r, cand_labels)
        mapping[r] = best if score >= threshold else None
    return mapping

def f1(precision: float, recall: float) -> float:
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def bridge_score(reference: Dict[str, Any], parsed: ParsedRDF) -> Dict[str, Any]:
    ref_nodes = sorted({n["label"].strip().lower() for n in reference.get("nodes", []) if n.get("label")})
    ref_edges = reference.get("edges", [])

    cand_nodes = sorted(set(parsed.labels.values()))  # labels from RDF
    cand_edges = rdf_edges_as_label_triples(parsed)

    node_map = align_labels(ref_nodes, cand_nodes, threshold=LABEL_MATCH_THRESHOLD)

    matched_ref_nodes = [r for r, c in node_map.items() if c is not None]
    node_recall = len(matched_ref_nodes) / max(1, len(ref_nodes))

    matched_cand = set(node_map[r] for r in matched_ref_nodes if node_map[r] is not None)
    node_precision = len(matched_cand) / max(1, len(cand_nodes))
    node_f1 = f1(node_precision, node_recall)

    cand_edge_set = set((s, p, o) for s, p, o in cand_edges)

    matched_edges = 0
    for e in ref_edges:
        s = (e.get("source") or "").strip().lower()
        t = (e.get("target") or "").strip().lower()
        rel = (e.get("relation") or "arrow").strip().lower()

        s2 = node_map.get(s)
        t2 = node_map.get(t)
        if not s2 or not t2:
            continue

        found = False
        for (cs, cp, co) in cand_edge_set:
            if cs == s2 and co == t2:
                if max(fuzz.partial_ratio(rel, cp.lower()), fuzz.partial_ratio("arrow", cp.lower())) >= 70:
                    found = True
                    break
        if found:
            matched_edges += 1

    edge_recall = matched_edges / max(1, len(ref_edges))
    edge_precision = matched_edges / max(1, len(cand_edges))
    edge_f1 = f1(edge_precision, edge_recall)

    halluc_nodes = [c for c in cand_nodes if _best_match(c, ref_nodes)[1] < LABEL_MATCH_THRESHOLD]
    halluc_rate = len(halluc_nodes) / max(1, len(cand_nodes))

    overall = (
        WEIGHTS["syntax_ok"] * (1.0 if parsed.syntax_ok else 0.0)
        + WEIGHTS["node_f1"] * node_f1
        + WEIGHTS["edge_f1"] * edge_f1
        - WEIGHTS["halluc_penalty"] * halluc_rate
    )
    overall = max(0.0, min(1.0, overall))

    return {
        "ref_nodes": len(ref_nodes),
        "ref_edges": len(ref_edges),
        "cand_nodes": len(cand_nodes),
        "cand_edges": len(cand_edges),
        "node_precision": node_precision,
        "node_recall": node_recall,
        "node_f1": node_f1,
        "edge_precision": edge_precision,
        "edge_recall": edge_recall,
        "edge_f1": edge_f1,
        "halluc_node_rate": halluc_rate,
        "syntax_ok": parsed.syntax_ok,
        "overall_det_bridge_score": overall,
        "halluc_nodes_sample": halluc_nodes[:15],
    }


In [ ]:
# =========================
# 8) LLM-as-a-judge prompts (Bridge + Direct)
# =========================

BRIDGE_JUDGE_PROMPT_TMPL = \"\"\"
You are an expert evaluator for RDF (Turtle) extracted from a diagram.

You are given:
(1) A reference diagram graph extracted from the image (JSON).
(2) A candidate RDF/Turtle output.

Evaluate the candidate for:
- faithfulness: does it match the reference (no hallucinated nodes/relations)?
- completeness: does it cover most nodes/edges?
- structural quality: correct directionality, reasonable predicates (skos:narrower/broader or clear custom properties).
- label fidelity: prefLabel matches node text (lowercase, exact-ish).

Return ONLY JSON with this schema:
{
  "faithfulness": 1-5,
  "completeness": 1-5,
  "structure": 1-5,
  "labels": 1-5,
  "overall": 1-5,
  "major_issues": ["..."],
  "notes": "short"
}

Reference graph JSON:
{reference_json}

Candidate RDF/Turtle:
{candidate_turtle}
\"\"\"

DIRECT_JUDGE_PROMPT_TMPL = \"\"\"
You are an expert evaluator for RDF (Turtle) extracted from a diagram image.

Task:
Given the image and candidate RDF/Turtle, score quality.

Criteria:
- Faithfulness to the image (no hallucinations)
- Completeness (covers most nodes and arrows)
- Correct directionality and semantics of relations
- SKOS usage and predicate naming quality
- Valid Turtle syntax (or close)

Return ONLY JSON:
{
  "faithfulness": 1-5,
  "completeness": 1-5,
  "structure": 1-5,
  "labels": 1-5,
  "syntax": 1-5,
  "overall": 1-5,
  "major_issues": ["..."],
  "notes": "short"
}
\"\"\"

def parse_json_loose(txt: str) -> Dict[str, Any]:
    txt = txt.strip()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        start = txt.find("{")
        end = txt.rfind("}")
        if start >= 0 and end > start:
            return json.loads(txt[start:end+1])
        raise

def llm_bridge_judge(model: str, reference: Dict[str, Any], turtle: str, temperature: float = 0.1) -> Dict[str, Any]:
    prompt = BRIDGE_JUDGE_PROMPT_TMPL.format(reference_json=json.dumps(reference, ensure_ascii=False), candidate_turtle=turtle)
    out = gemini_text(model=model, prompt=prompt, temperature=temperature, max_output_tokens=2048)
    return parse_json_loose(out)

def llm_direct_judge(model: str, image_path: str, turtle: str, temperature: float = 0.1) -> Dict[str, Any]:
    prompt = DIRECT_JUDGE_PROMPT_TMPL + \"\\n\\nCandidate RDF/Turtle:\\n\" + turtle
    out = gemini_multimodal(model=model, image_path=image_path, prompt=prompt, temperature=temperature, max_output_tokens=2048)
    return parse_json_loose(out)


In [ ]:
# =========================
# 9) Multi-judge aggregation
# =========================

def median(xs: List[float]) -> float:
    xs2 = sorted(xs)
    n = len(xs2)
    if n == 0:
        return float("nan")
    if n % 2 == 1:
        return float(xs2[n//2])
    return float((xs2[n//2 - 1] + xs2[n//2]) / 2)

def aggregate_judges(judge_outputs: List[Dict[str, Any]]) -> Dict[str, Any]:
    keys = ["faithfulness", "completeness", "structure", "labels", "syntax", "overall"]
    agg = {}
    for k in keys:
        vals = [j.get(k) for j in judge_outputs if isinstance(j.get(k), (int, float))]
        if vals:
            agg[k + "_median"] = median(vals)
            agg[k + "_min"] = float(min(vals))
            agg[k + "_max"] = float(max(vals))
            agg[k + "_range"] = float(max(vals) - min(vals))
        else:
            agg[k + "_median"] = None
            agg[k + "_range"] = None

    issues = []
    for j in judge_outputs:
        for it in j.get("major_issues", []) if isinstance(j.get("major_issues"), list) else []:
            if it not in issues:
                issues.append(it)
    agg["major_issues_union"] = issues[:25]
    return agg


In [ ]:
# =========================
# 10) Run evaluation
# =========================

def evaluate_row(row: pd.Series, mode: str = "bridge") -> Dict[str, Any]:
    image_path = resolve_image_path(row["source_image"])
    turtle = row["rdf_graph_turtle"]
    parsed = parse_turtle(turtle)

    result = {
        "run": row["run"],
        "image_key": row["image_key"],
        "source_image": row["source_image"],
        "resolved_image_path": image_path,
        "syntax_ok": parsed.syntax_ok,
        "n_triples": parsed.n_triples,
        "n_labels": len(parsed.labels),
    }

    if mode == "bridge":
        ref = get_bridge_reference(image_path)
        det = bridge_score(ref, parsed)
        result.update({f"det_{k}": v for k, v in det.items()})

        judge_outputs = []
        for jc in JUDGES:
            try:
                j = llm_bridge_judge(model=jc["model"], reference=ref, turtle=turtle, temperature=jc["temperature"])
                j["judge_name"] = jc["name"]
                j["judge_model"] = jc["model"]
                judge_outputs.append(j)
            except Exception as e:
                judge_outputs.append({"judge_name": jc["name"], "judge_model": jc["model"], "error": str(e)})
        result["bridge_judges"] = judge_outputs
        result.update({f"bridge_{k}": v for k, v in aggregate_judges(judge_outputs).items()})

    elif mode == "direct":
        judge_outputs = []
        for jc in JUDGES:
            try:
                j = llm_direct_judge(model=jc["model"], image_path=image_path, turtle=turtle, temperature=jc["temperature"])
                j["judge_name"] = jc["name"]
                j["judge_model"] = jc["model"]
                judge_outputs.append(j)
            except Exception as e:
                judge_outputs.append({"judge_name": jc["name"], "judge_model": jc["model"], "error": str(e)})
        result["direct_judges"] = judge_outputs
        result.update({f"direct_{k}": v for k, v in aggregate_judges(judge_outputs).items()})

    else:
        raise ValueError("mode must be 'bridge' or 'direct'")

    return result

# ---- Run on a small sample first ----
sample = df_all.sample(min(3, len(df_all)), random_state=42).reset_index(drop=True)
sample_results = []
for _, r in tqdm(sample.iterrows(), total=len(sample)):
    sample_results.append(evaluate_row(r, mode="bridge"))

pd.DataFrame([{k:v for k,v in x.items() if k not in ("bridge_judges","direct_judges")} for x in sample_results]).head()


In [ ]:
# ---- Full run (can take time/cost depending on dataset size) ----
# Tip: start with a subset (e.g., head(50)) and scale up.

EVAL_MODE = "bridge"  # "bridge" or "direct"
MAX_ITEMS = None      # set e.g. 200 to limit

to_eval = df_all.copy()
if MAX_ITEMS is not None:
    to_eval = to_eval.head(MAX_ITEMS)

results = []
for _, r in tqdm(to_eval.iterrows(), total=len(to_eval)):
    results.append(evaluate_row(r, mode=EVAL_MODE))

Path("evaluation_results.json").write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")

flat = pd.DataFrame([{k:v for k,v in x.items() if k not in ("bridge_judges","direct_judges")} for x in results])
flat.to_csv("evaluation_results.csv", index=False)

flat.head()


In [ ]:
# =========================
# 11) Summaries
# =========================
flat = pd.read_csv("evaluation_results.csv")

if EVAL_MODE == "bridge":
    summary_cols = [
        "det_overall_det_bridge_score",
        "bridge_overall_median",
        "bridge_overall_range",
        "det_node_f1",
        "det_edge_f1",
        "det_halluc_node_rate",
        "syntax_ok",
    ]
else:
    summary_cols = [
        "direct_overall_median",
        "direct_overall_range",
        "syntax_ok",
    ]

summary = flat.groupby("run")[summary_cols].mean(numeric_only=True).reset_index()
summary


In [ ]:
# Optional: show worst cases by deterministic bridge score
if EVAL_MODE == "bridge" and "det_overall_det_bridge_score" in flat.columns:
    flat.sort_values("det_overall_det_bridge_score").head(20)[
        ["run","image_key","det_overall_det_bridge_score","det_node_f1","det_edge_f1","det_halluc_node_rate","syntax_ok"]
    ]


In [ ]:
# =========================
# 12) Inspect one example in detail
# =========================
example_key = flat.sort_values("det_overall_det_bridge_score").iloc[0]["image_key"] if EVAL_MODE=="bridge" else flat.iloc[0]["image_key"]
example_run = flat[flat["image_key"] == example_key].iloc[0]["run"]

row = df_all[(df_all["image_key"] == example_key) & (df_all["run"] == example_run)].iloc[0]
img_path = resolve_image_path(row["source_image"])
print("Example:", example_run, example_key)
print("Image path:", img_path)

ref = get_bridge_reference(img_path)
print(json.dumps(ref, indent=2, ensure_ascii=False)[:2000], "...\n")

parsed = parse_turtle(row["rdf_graph_turtle"])
print("Syntax OK:", parsed.syntax_ok, "| triples:", parsed.n_triples, "| labels:", len(parsed.labels))
print("Some labels:", list(parsed.labels.values())[:15])
